In [1]:
import pandas as pd
import numpy as np
import glob, os



In [7]:
path = r'd:\USYD\DATA3888\group_asm\Optiver\individual_book_train'
files = glob.glob(os.path.join(path, "*.csv"))
print(files) 


['d:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_0.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_1.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_10.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_100.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_101.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_102.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_103.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_104.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_105.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_107.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_108.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\individual_book_train\\stock_109.csv', 'd:\\USYD\\DATA3888\\group_asm\\Optiver\\ind

In [11]:
def aggregate_stock(f):
    df = pd.read_csv(f)

    # Compute WAP and BidAskSpread
    df['WAP'] = (df['bid_price1'] * df['ask_size1'] + df['ask_price1'] * df['bid_size1']) / \
                (df['bid_size1'] + df['ask_size1'])
    df['BidAskSpread'] = df['ask_price1'] / df['bid_price1'] - 1

    # Compute log return within each time_id
    df = df.sort_values(['time_id', 'seconds_in_bucket'])
    df['log_return'] = df.groupby('time_id')['WAP'].transform(
        lambda x: np.log(x / x.shift(1))
    ).fillna(0)

    # Assign 30s bucket
    df['time_bucket'] = np.ceil(df['seconds_in_bucket'] / 30).astype(int)

    # Aggregate per time_id + 30s bucket
    agg = df.groupby(['stock_id', 'time_id', 'time_bucket']).agg(
        WAP_mean      = ('WAP', 'mean'),
        BidAskSpread_mean = ('BidAskSpread', 'mean'),
        volatility    = ('log_return', lambda x: np.sqrt(np.sum(x**2)))
    ).reset_index()

    return agg  # ~10,000 rows instead of ~1,500,000

# Process one file at a time, never loading all raw data at once
all_stocks = pd.concat([aggregate_stock(f) for f in files], ignore_index=True)

print(all_stocks.shape)
print(f"Memory: {all_stocks.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Save the merged aggregated file — tiny!
all_stocks.to_csv(
    r'd:\USYD\DATA3888\group_asm\optiver_aggregated.csv',
    index=False
)
print("Done!")

KeyboardInterrupt: 

In [ ]:
''' 
These line of code will be commented out since I already merged all file. 
Basically, 126 csv files has been aggregated as week 3's material instruction indicated, the new merged file will contain stock information, WAP, BAS, and
log return.
'''
# import duckdb

# path = r'd:\USYD\DATA3888\group_asm\Optiver\individual_book_train'

# conn = duckdb.connect()

# result = conn.execute(f"""
#     SELECT
#         stock_id,
#         time_id,
#         CEIL(seconds_in_bucket / 30.0) AS time_bucket,
#         AVG((bid_price1 * ask_size1 + ask_price1 * bid_size1) / 
#             (bid_size1 + ask_size1)) AS WAP_mean,
#         AVG(ask_price1 / bid_price1 - 1) AS BidAskSpread_mean,
#         SQRT(SUM(POWER(
#             LN((bid_price1 * ask_size1 + ask_price1 * bid_size1) / 
#                (bid_size1 + ask_size1)), 2)
#         )) AS volatility
#     FROM read_csv_auto('{path}/*.csv')
#     GROUP BY stock_id, time_id, time_bucket
#     ORDER BY stock_id, time_id, time_bucket
# """).df()  # returns a pandas dataframe

# result.to_csv(r'd:\USYD\DATA3888\group_asm\optiver_aggregated.csv', index=False)
# print(result.shape)

(8998663, 6)


In [ ]:
size_mb = os.path.getsize(r'd:\USYD\DATA3888\group_asm\optiver_aggregated.csv') / (1024**2)
print(f"File size: {size_mb:.2f} MB")

#These code help you check if the current file is overloading your computer, I've saved your laptop :)
df = pd.read_csv(r'd:\USYD\DATA3888\group_asm\optiver_aggregated.csv')
mem_mb = df.memory_usage(deep=True).sum() / (1024**2)

print(f"Shape: {df.shape}")
print(f"RAM usage: {mem_mb:.2f} MB")
print(df.head())

File size: 656.03 MB
Shape: (8998663, 6)
RAM usage: 411.93 MB
   stock_id  time_id  time_bucket  WAP_mean  BidAskSpread_mean  volatility
0         0        5          0.0  1.001434           0.000878    0.001433
1         0        5          1.0  1.001601           0.000957    0.006366
2         0        5          2.0  1.002978           0.000748    0.009511
3         0        5          3.0  1.003786           0.000876    0.011409
4         0        5          4.0  1.004046           0.000873    0.016157
